<a href="https://colab.research.google.com/github/Dewdahara/CRMCM_CW1_w2054873/blob/main/CRMCM_CW1_w2054873.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6BUIS017W CRM & CM with Business Intelligence: Individual Coursework

## Task 1: Data Preparation and Feature Engineering

### Setup

 **Data snapshot update (10 November 2026):** Constituents and daily Close, Adjusted Close and Volume data are stored together in one frozen Excel workbook. Subsequent runs load this workbook and do not refresh Yahoo Finance prices. The benchmark (^GSPC) is included in the price worksheets. The current constituent list remains a point-in-time universe, not a historical membership archive; results are subject to survivorship and constituent-history bias.

In [1]:
# Install the extra packages: yfinance for the prices, lxml so pandas can read the Wikipedia table,
# and seaborn for the charts.
%pip install -q yfinance lxml seaborn

In [2]:
# Import the libraries and record the package versions used in this run.
import os, json, platform, warnings
from datetime import datetime, timezone
from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display
import scipy
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

PACKAGE_VERSIONS = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "yfinance": yf.__version__,
    "matplotlib": matplotlib.__version__,
    "requests": requests.__version__,
    "seaborn": sns.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__
}
PACKAGE_VERSIONS

{'python': '3.13.18',
 'pandas': '2.2.3',
 'numpy': '2.4.6',
 'yfinance': '0.2.66',
 'matplotlib': '3.10.0',
 'requests': '2.32.4',
 'seaborn': '0.13.2',
 'scipy': '1.16.3',
 'scikit-learn': '1.6.1'}

In [3]:
# Settings for the study period and the data checks. The end date is exclusive, so the last day used
# is 31 Dec 2024.
START_DATE = "2021-01-01"
END_DATE = "2025-01-01"
BENCHMARK = "^GSPC"
BASE_WINDOW = 20
ACTIVITY_WINDOW = 60
HIGH_VOL_MULT = 2.0
MIN_AVAILABILITY = 0.80
MIN_WEEKS = 52

# Prices are never filled in, so dates before a listing stay missing.
FILL_PRICES = False
# Setting this to True downloads fresh constituents and prices and replaces the workbook.
# Once the workbook is saved, set it back to False for every later run.
CREATE_DATA_SNAPSHOT = False
SNAPSHOT_XLSX = "sp500_market_data_snapshot.xlsx"
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)